# IOAI — 2025 Selection Camp Hotspot (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/Satellite_Images-1'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-hotspot.zip', '/tmp/hs.zip')
    with zipfile.ZipFile('/tmp/hs.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/Satellite_Images-1' if os.path.exists('data/Satellite_Images-1') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

Rezolvare de 96/100

Este simpla si scurta, fiind bazata doar pe procesare de imagini cu OpenCV, fara niciun model utilizat.

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import cv2

import os

In [ ]:
seed = 42
root_path = "data"   # 로컬 staged 데이터(원본: /home/stefan/.../kits/roai-2025/hotspot)

# Data preparation

In [ ]:
def image_to_rle(img_arr: np.array) -> list[int]:
    side, side = img_arr.shape[0], img_arr.shape[1]
    img_arr = img_arr.reshape(side*side, order='F')
    
    start = 0
    length = 0
    res = []
    for i in range(side * side):
        if img_arr[i] == 255:
            if length == 0:
                start = i + 1
            length = length + 1
        else:
            if length > 0:
                res.append(start)
                res.append(length)
            length = 0
    if length > 0:
        res.append(start)
        res.append(length)
        
    return res

In [ ]:
def get_subtask_images(subtask_id: int) -> list[np.array]:
    path = f"{root_path}/Satellite_Images-{subtask_id}"
    images = os.listdir(path)
    
    arr = []
    for image_path in images:
        img = np.array(Image.open(f"{path}/{image_path}")) # .shape = (256, 256, 3)
        arr.append({"img": img, "filename": image_path})
    
    return arr

# Image processing

In [ ]:
subtask4_images = get_subtask_images(4)

In [ ]:
def get_mask(img: np.array, subtask_id: int = 3) -> np.array:
    grayscale = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blur = cv2.medianBlur(grayscale, 7)

    threshold_value = 60 if subtask_id != 4 else 80
    _, threshold = cv2.threshold(blur, threshold_value, 255, cv2.THRESH_BINARY)
    return threshold # .shape = (256, 256)

In [ ]:
plt.figure()
f, axarr = plt.subplots(1, 2) 

img = subtask4_images[11]["img"]
masked = get_mask(img)

axarr[0].imshow(img)
axarr[1].imshow(masked, cmap='gray')

In [ ]:
# is the format good for RLE?
len(image_to_rle(masked)) > 0

# Submission

In [ ]:
subtasks = [{"masked": [], "id": []} for _ in range(4)]

In [ ]:
for subtask_id in range(4):
    subtask_images = get_subtask_images(subtask_id+1)

    for image in tqdm(subtask_images):
        masked = get_mask(image["img"], subtask_id+1)
        masked_rle = image_to_rle(masked)

        subtasks[subtask_id]["masked"].append(masked_rle)
        subtasks[subtask_id]["id"].append(image["filename"])

In [ ]:
subtasks_df = [
    pd.DataFrame({
        "subtaskID": i+1,
        "datapointID": subtask["id"],
        "answer": subtask["masked"],
    }) for i, subtask in enumerate(subtasks)
]

submission = pd.concat(subtasks_df, ignore_index=True)

In [ ]:
submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)